In [1]:
using DifferentialEquations # for the actual time evolution
using OrdinaryDiffEq # for ODEs
using Plots # for plotting
using Base.Threads # for parallelization
using StaticArrays # somehow needed to use multiple variables in DifferentialEquations.jl

using Plots, LaTeXStrings, Colors
using Plots.PlotMeasures
using LinearAlgebra

using Random, Distributions

using FFTW # discrete Fourier transform

using JLD2 # for file saving

In [3]:
level = "../../../../"

include(joinpath(level, "src/4th-order-FD-stencils.jl"));
include(joinpath(level, "src/evolution_Liouville_larger_cutoff.jl"));
include(joinpath(level, "src/hamiltonian_Liouville.jl"));
include(joinpath(level, "src/initial_data_waves.jl"));
include(joinpath(level, "src/visualisation.jl"));

### evolution

In [3]:
function artisan_evolution_at_resolution(Nx, stableRandomSeed, pModel, pInit, target_time)
    # unpack model parameters
    (mphi2, mchi2, c4, c, epsDiss) = pModel;
    
    # set the spatial discretization
    NboundaryPadding = 2;#Int(div(Nx,2));  # Number of boundary padding points
    dx = 1/(Nx);  # Grid spacing
        
    pGrid = (dx, Nx, NboundaryPadding);
    # reset a combined set of parameters (residual from old structure ... could be modified)
    # TODO: modify to pGrid, pModel, pInit
    p = (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding);

    # set the time span
    tspan = (0, target_time);

    # set the evolution method
    time_integration_method = RK4();
    
    # generate initial conditions
    u0 = initial_data(
        range(0, step=dx, length=(Nx + 2 * NboundaryPadding)),
        p, 
        pInit
    );
        
    # set the problem
    prob = ODEProblem(finite_differenced_pde_with_bc!, u0, tspan, p);

    sol = solve(
        prob, time_integration_method, 
        saveat = tspan[end]/10^3, #exp.(range(log(tspan[1]), log(tspan[end]), length=10^4))
        dt=dx/10, 
        adaptive = false, 
        dense=false, 
        maxiters=typemax(Int),
        callback=field_size_callback
    );
        
    # obtain the hamiltonian
    hamiltonian = zeros(length(sol.u))
    hamphi = zeros(length(sol.u))
    hamchi = zeros(length(sol.u))
    for i = 1:length(sol.u)
        hamiltonian[i] = nintegrate_simps(hamiltonian_density(sol.u[i], p), dx)
        hamphi[i] = nintegrate_simps(hamiltonian_phi(sol.u[i], p), dx)
        hamchi[i] = nintegrate_simps(hamiltonian_chi(sol.u[i], p), dx)
    end
    
    return (p, sol, hamiltonian, hamphi, hamchi)
end

artisan_evolution_at_resolution (generic function with 1 method)

In [4]:
function evolution_at_amplitude(amp, current_target_time, current_res_log2, stableRandomSeed)
    
    #stableRandomSeed = 42
    print("persistent random seed: ", stableRandomSeed, "\n")
    
    print("current characteristic amplitude: ", amp, "\n")
    
    # set monitoring flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # parameters of the model
    mphi2 = 1;
    mchi2 = 15;
    c4 = 1;
    c = -1.;
    epsDiss = 0;

    # parameters of the initial data
    a0phi = 0; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    a0chi = a0phi; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    k0phi = 1;
    k0chi = 2 * k0phi;
    x0phi = 0;
    x0chi = 1/3;

    offsetphi = sqrt(((mphi2 - mchi2)^2 - 8 * mphi2 * c4) / (32 * c4^2))
    offsetchi = - sqrt(((mphi2 - mchi2)^2 - 8 * mchi2 * c4) / (32 * c4^2))

    aStochastic = amp;
    mink = 1;
    maxk = 4;
    
    desiredTkinPhi = NaN;
    desiredTkinChi = NaN;
    
    # set the combined set of parameters 
    pModel = (mphi2, mchi2, c4, c, epsDiss);
    pInit = (
        a0phi, a0chi, k0phi, k0chi, x0phi, x0chi, 
        offsetphi, offsetchi, 
        aStochastic, mink, maxk, stableRandomSeed,
        desiredTkinPhi, desiredTkinChi
    );
     
    # set some tables to store intermediate output
    resTab = [2^i for i in current_res_log2-2:current_res_log2]
    pTab = []
    solTab = []
    hamiltonianTab = []
    hamPhiTab = []
    hamChiTab = []
    
    #############################
    # evolution
    #############################
    
    # run evolution
    for res in resTab
        print("current resolution: ", res, "\n")
        # run the evolution
        @time (p, sol, hamiltonian, hamPhi, hamChi) = artisan_evolution_at_resolution(
            res, stableRandomSeed, pModel, pInit, current_target_time
        )
        print("... terminated", "\n")
        # unpack parameters
        (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding) = p
        # append the results
        push!(pTab, p)
        push!(solTab, sol)
        push!(hamiltonianTab, hamiltonian)
        push!(hamPhiTab, hamPhi)
        push!(hamChiTab, hamChi)
    end
    
    #############################
    # CONVERGENCE
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",amp)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        print("Output plot directory does not exist. Creating it ...\n")
        mkpath(dir_path)
    else
        print("Output plot directory already exists.\n")
    end
    
    # plot and determine convergence 
    loss_of_convergence_time = save_convergence_plots(
        resTab, pTab, solTab, 
        hamiltonianTab, 
        dir_path
    )
    if loss_of_convergence_time >= solTab[end].t[end]
        print("Convergence kept at all times.\n")
    else
        print("Convergence lost at time t=",loss_of_convergence_time,"\n")
    end
    
    # determine the index of convergence loss
    loss_of_convergence_index = findfirst(t -> t > loss_of_convergence_time, solTab[end].t)
    if loss_of_convergence_index === nothing
        loss_of_convergence_index = length(solTab[end].t)
    end
    
    #print(10 * hamPhiTab[end][1],"\n")
    #print(hamPhiTab[end][2:10],"\n")
    
    # determine the onset time of the runaway (10-fold increase in either kinetic energy)
    runaway_index_phi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamPhiTab[end][1:Int(div(length(hamPhiTab[end]),4))+1])), 
        hamPhiTab[end]
    )
    if runaway_index_phi === nothing
        runaway_index_phi = length(hamPhiTab[end])
    end
    runaway_index_chi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamChiTab[end][1:Int(div(length(hamChiTab[end]),4))+1])), 
        hamChiTab[end]
    )
    if runaway_index_chi === nothing
        runaway_index_chi = length(hamChiTab[end])
    end
    runaway_index = min(runaway_index_phi, runaway_index_chi)
    runaway_time = solTab[end].t[runaway_index]
    if runaway_time >= solTab[end].t[end]
        print("No runaway detected.\n")
    else
        print("Runaway detected at time t=",runaway_time,"\n")
    end    
    
    #############################
    # GENERATE REMAINING PLOTS IF DESIRED
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",amp)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        #print("Directory does not exist. Creating it...")
        mkpath(dir_path)
    end
        
    # plot energy components
    save_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_normalised_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_difference_in_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    
    # plot field heatmaps
    save_density_plots(
        solTab[end], pTab[end], pInit,
        dir_path,
        loss_of_convergence_time=loss_of_convergence_time
    )
    
    # save snapshots
    save_snaps(
        solTab[end], pTab[end];
        snap_intervals=Int(round(length(solTab[end])/1)), 
        yrangeVal=1.2,
        dir_path = dir_path
    );
    
    # animate the fields
    save_animation(
        solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
        pTab[end],
        join([dir_path, "/animation_Nx=", resTab[end], ".gif"])
    );    
#     # animate frequencies
#     save_animation_momentum_space(
#         solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
#         pTab[end],
#         join([dir_path, "/animation_momentum_space_Nx=", resTab[end], ".gif"])
#     );
    
    print("Finished plotting.", "\n")
    
    #############################
    # SAVE DATA
    #############################
    
    if runaway_time > loss_of_convergence_time
        print("WARNING: Convergence not maintained until onset of runaway. Resolution insufficient.", "\n")
    else
        convergence_maintained = true
        if runaway_time >= solTab[end].t[end]
            print("WARNING: Lower bound only because target time insufficient.", "\n")
        else
            lower_bound_only = false
        end
    end
    
    dir_path = string("dat/",stableRandomSeed)
    if !isdir(dir_path)
        mkpath(dir_path)
    end

    timesteps = solTab[end].t
    stable_until = min(runaway_time, loss_of_convergence_time)

    @save joinpath(pwd(), dir_path, string(amp,".jld2")) amp stable_until lower_bound_only timesteps hamiltonianTab hamPhiTab hamChiTab

    print("Saved data.", "\n")

    
    return (runaway_time, convergence_maintained, lower_bound_only)
end

evolution_at_amplitude (generic function with 1 method)

### main()

In [5]:
amp_base = 1.2
amplitudes = reverse([amp_base^i for i in -8:8])

17-element Vector{Float64}:
 4.2998169599999985
 3.583180799999999
 2.9859839999999993
 2.4883199999999994
 2.0736
 1.728
 1.44
 1.2
 1.0
 0.8333333333333334
 0.6944444444444445
 0.5787037037037037
 0.4822530864197532
 0.40187757201646096
 0.3348979766803842
 0.2790816472336535
 0.2325680393613779

In [6]:
    amplitudes = [invAmp for invAmp in 0.5:0.5:16]
    amplitudes = amplitudes.^(-1)

32-element Vector{Float64}:
 2.0
 1.0
 0.6666666666666666
 0.5
 0.4
 0.3333333333333333
 0.2857142857142857
 0.25
 0.2222222222222222
 0.2
 0.18181818181818182
 0.16666666666666666
 0.15384615384615385
 ⋮
 0.09523809523809523
 0.09090909090909091
 0.08695652173913043
 0.08333333333333333
 0.08
 0.07692307692307693
 0.07407407407407407
 0.07142857142857142
 0.06896551724137931
 0.06666666666666667
 0.06451612903225806
 0.0625

In [7]:
function main()
    
    # some random seed (can be modified at will)
    stableRandomSeed = rand(1:10^7)
    
    # initialise flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # set abort criteria ...
    highest_res_log2 = 13;
    max_target_time = 8 * 10^3;
    # ... and their initial values
    current_res_log2 = 11;
    current_target_time = 1;
    
    # initialise the runaway time for handover to next amplitude
    runaway_time = Inf;
    
    # set table of desired amplitudes (NOTE: links to scaling assumption below)
    amp_base = 1
    amplitudes = [invAmp for invAmp in 0.5:0.5:16]
    amplitudes = amplitudes.^(-1)
    
    # loop over all amplitudes
    for amp in amplitudes
        
        # re-attempt while flags not positive or until abort criteria met
        while (!convergence_maintained||lower_bound_only) && (current_res_log2 <= highest_res_log2) && (current_target_time <= max_target_time)
            # attempt run and obtain flags
            (runaway_time, convergence_maintained, lower_bound_only) = evolution_at_amplitude(
                amp, 
                current_target_time, 
                current_res_log2,
                stableRandomSeed
            )
            # update according to obtained flags
            if convergence_maintained                
                if lower_bound_only
                    current_target_time = current_target_time * 4
                    print("Increasing target time to T = ", current_target_time, "\n")
                else
                    current_target_time = min(runaway_time, current_target_time);
                    print("Target time reset to confidently detected runaway time T = ", current_target_time, "\n")
                end
            else
                current_res_log2 = current_res_log2 + 1;
                print("Increasing resolution from N = ", current_res_log2 - 1, " to ", current_res_log2, "\n")
            end
        end
        
        print("AMPLITUDE A = ", amp, " DONE!\n")
        
        # update target time based on the presumed scaling assumption and adapt the target time accordingly
        current_target_time = runaway_time
        print("Updating target time for next amplitude from T = ", current_target_time, " ... ")
        current_target_time = current_target_time * exp(amp_base)     
        print("to T = ", current_target_time, "\n")
        
        # decrease resolution if convergence was maintained in previous step
#         if convergence_maintained
#             current_res_log2 = current_res_log2 - 1;
#             print("Decreasing resolution from N = ", current_res_log2 + 1, " to ", current_res_log2, "\n")
#         end
        
        # check whether it makes sense to go on; otherwise abort 
        if convergence_maintained && lower_bound_only && current_target_time >= max_target_time
            print("ABORT: maximum target time approached in converged simulation; no use to proceed")
            return
        end
        
        # ensure that current params don't exceed the abort criteria for the next step
        current_res_log2 = min(current_res_log2, highest_res_log2)
        current_target_time = min(current_target_time, max_target_time)
        
        # reset the flags
        convergence_maintained = false;
        lower_bound_only = true;
    end

end

main (generic function with 1 method)

In [ ]:
main()

persistent random seed: 8073856
current characteristic amplitude: 2.0
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.3983615101693099
		max |amplitude| chi before rescaling: 1.806057912681021
  4.558113 seconds (3.61 M allocations: 1.286 GiB, 4.29% gc time, 86.41% compilation time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.398431219738624
		max |amplitude| chi before rescaling: 1.8060643629254758
  2.103241 seconds (779.76 k allocations: 4.025 GiB, 27.01% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.398431219738624
		max |amplitude| chi before rescaling: 1.806073163994798
  6.461074 seconds (2.18 M allocations: 15.496 GiB, 5.64% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence kept at all times.
No runaway detected.
Finished plotting.


[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/03_rand/plots/8073856/2.0/animation_Nx=2048.gif


Saved data.
Increasing target time to T = 4
persistent random seed: 8073856
current characteristic amplitude: 2.0
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.3983615101693099
		max |amplitude| chi before rescaling: 1.806057912681021
  2.759459 seconds (1.52 M allocations: 4.037 GiB, 26.09% gc time, 0.43% compilation time: 100% of which was recompilation)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.398431219738624
		max |amplitude| chi before rescaling: 1.8060643629254758
  7.245007 seconds (2.99 M allocations: 15.569 GiB, 6.51% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.398431219738624
		max |amplitude| chi before rescaling: 1.806073163994798
 25.028841 seconds (8.57 M allocations: 60.935 GiB, 9.24% gc time)
... terminated
Output plot directory already exists.
Convergence kept at all times.
No 

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/03_rand/plots/8073856/2.0/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 9.59101562500065.
  5.000874 seconds (3.57 M allocations: 9.515 GiB, 7.14% gc time, 0.45% compilation time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.398431219738624
		max |amplitude| chi before rescaling: 1.8060643629254758
Terminating because one of the fields grew too large at time t = 9.557910156267651.
 15.307169 seconds (7.07 M allocations: 36.882 GiB, 5.55% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.398431219738624
		max |amplitude| chi before rescaling: 1.806073163994798
Terminating because one of the fields grew too large at time t = 9.48798828124889.
 56.157795 seconds (20.24 M allocations: 143.918 GiB, 4.45% gc time)
... terminated
Output plot directory already exists.
Convergence lost at time t=7.68
Runaway detected at time t=4.784
Finished plotting.
Saved data.
Targ

...

Terminating because one of the fields grew too large at time t = 159.5820312505235.
 76.786820 seconds (58.86 M allocations: 157.457 GiB, 11.25% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.11653593497821867
		max |amplitude| chi before rescaling: 0.15050536357712294
Terminating because one of the fields grew too large at time t = 159.50019531740142.
260.797570 seconds (117.63 M allocations: 613.813 GiB, 9.05% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.11653593497821867
		max |amplitude| chi before rescaling: 0.15050609699956655
Terminating because one of the fields grew too large at time t = 159.7162109353747.
939.328586 seconds (340.24 M allocations: 2.363 TiB, 9.82% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=151.56312301125124
Runaway detected at time t=135.15019598465213


[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/03_rand/plots/8073856/0.16666666666666666/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 187.9269531271734.
 96.871194 seconds (69.30 M allocations: 185.405 GiB, 10.84% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.10757163228758647
		max |amplitude| chi before rescaling: 0.1389280279173443
Terminating because one of the fields grew too large at time t = 188.2193359440731.
319.868467 seconds (138.80 M allocations: 724.295 GiB, 8.74% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.10757163228758647
		max |amplitude| chi before rescaling: 0.13892870492267684
Terminating because one of the fields grew too large at time t = 188.25468749123007.
1154.950136 seconds (401.02 M allocations: 2.785 TiB, 9.35% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=186.62717150374132
Runaway detected at time t=169.72786069828442


[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/03_rand/plots/8073856/0.15384615384615385/animation_Nx=2048.gif


user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.09988296501209355
		max |amplitude| chi before rescaling: 0.1290041366200729
Terminating because one of the fields grew too large at time t = 211.1953125035278.
109.539930 seconds (77.88 M allocations: 208.350 GiB, 10.40% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.09988794426704456
		max |amplitude| chi before rescaling: 0.12900459735181974


### export .jl for production run

In [4]:
using NBInclude
nbexport("main.jl", "main.ipynb")